In [9]:
import requests

HEADERS = {"User-Agent": "UniMannheim-SMDA-Project/1.0 (student research project)"}

def test_categories():
    # Test a few different category names to see which ones have articles
    categories = [
        "Category:All_articles_with_topics_of_unclear_notability",
        "Category:Articles_with_specifically_marked_weasel-worded_phrases",
        "Category:All_Wikipedia_articles_written_in_American_English",  # just a sanity check
        "Category:Neutral_point_of_view_disputes",
        "Category:Wikipedia_neutral_point_of_view_disputes",
    ]
    
    for cat in categories:
        url = "https://en.wikipedia.org/w/api.php"
        params = {
            "action": "query",
            "list": "categorymembers",
            "cmtitle": cat,
            "cmlimit": 3,
            "cmtype": "page",
            "format": "json"
        }
        r = requests.get(url, params=params, headers=HEADERS)
        articles = r.json()["query"]["categorymembers"]
        print(f"\n{cat}")
        print(f"  → {len(articles)} articles found")
        for a in articles:
            print(f"     - {a['title']}")

test_categories()


Category:All_articles_with_topics_of_unclear_notability
  → 3 articles found
     - .22 ARC
     - .22 BR Remington
     - .45 Parabellum Bloodhound

Category:Articles_with_specifically_marked_weasel-worded_phrases
  → 0 articles found

Category:All_Wikipedia_articles_written_in_American_English
  → 3 articles found
     - -eaux
     - -maxxing
     - !!!

Category:Neutral_point_of_view_disputes
  → 0 articles found

Category:Wikipedia_neutral_point_of_view_disputes
  → 0 articles found


In [6]:
import requests

HEADERS = {"User-Agent": "UniMannheim-SMDA-Project/1.0 (student research project)"}

def test_article_content(title="Climate change"):
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "titles": title,
        "prop": "revisions|info",
        "rvprop": "content|ids",
        "rvslots": "main",
        "format": "json"
    }
    r = requests.get(url, params=params, headers=HEADERS)
    page = list(r.json()["query"]["pages"].values())[0]
    content = page["revisions"][0]["slots"]["main"]["*"]
    print(f"Title: {page['title']}")
    print(f"Content length: {len(content)} characters")
    print(f"First 300 chars:\n{content[:300]}")

test_article_content()

Title: Climate change
Content length: 331529 characters
First 300 chars:
{{Short description|Human-caused changes to climate on Earth}}
{{About|the human-induced rise in global temperatures|natural historical climate trends|Climate variability and change}}
{{Redirect|Global warming||Climate change (disambiguation)|and|Global warming (disambiguation)}}
{{Featured article}


In [10]:
import requests

HEADERS = {"User-Agent": "UniMannheim-SMDA-Project/1.0 (student research project)"}

def find_dispute_categories():
    # Search for categories containing "dispute" or "POV"
    url = "https://en.wikipedia.org/w/api.php"
    
    for search_term in ["dispute", "neutrality", "POV", "biased"]:
        params = {
            "action": "query",
            "list": "allcategories",
            "acprefix": f"Articles with",
            "aclimit": 10,
            "format": "json"
        }
        r = requests.get(url, params=params, headers=HEADERS)
        cats = r.json()["query"]["allcategories"]
        print(f"\nSearch: '{search_term}'")
        for c in cats:
            if any(word in c["*"].lower() for word in ["dispute", "pov", "neutral", "bias", "contested"]):
                print(f"  → Category:{c['*']}")

find_dispute_categories()


Search: 'dispute'

Search: 'neutrality'

Search: 'POV'

Search: 'biased'


In [11]:
import requests

HEADERS = {"User-Agent": "UniMannheim-SMDA-Project/1.0 (student research project)"}

def check_known_categories():
    # These are pulled directly from Wikipedia's dispute template documentation
    candidates = [
        "Category:NPOV_disputes_from_2024",
        "Category:NPOV_disputes_from_2023",
        "Category:All_NPOV_disputes",
        "Category:Articles_disputed_from_2024",
        "Category:Articles_disputed_from_2023",
        "Category:All_articles_disputed",
        "Category:Accuracy_disputes",
        "Category:Accuracy_disputes_from_2024",
        "Category:All_accuracy_disputes",
        "Category:POV_check",
    ]
    
    for cat in candidates:
        params = {
            "action": "query",
            "list": "categorymembers",
            "cmtitle": cat,
            "cmlimit": 3,
            "cmtype": "page",
            "format": "json"
        }
        r = requests.get("https://en.wikipedia.org/w/api.php", params=params, headers=HEADERS)
        articles = r.json()["query"]["categorymembers"]
        status = f"{len(articles)} articles" if articles else "EMPTY"
        print(f"{cat.replace('Category:', '')}: {status}")
        for a in articles:
            print(f"   - {a['title']}")

check_known_categories()

NPOV_disputes_from_2024: EMPTY
NPOV_disputes_from_2023: EMPTY
All_NPOV_disputes: EMPTY
Articles_disputed_from_2024: EMPTY
Articles_disputed_from_2023: EMPTY
All_articles_disputed: EMPTY
Accuracy_disputes: EMPTY
Accuracy_disputes_from_2024: EMPTY
All_accuracy_disputes: 3 articles
   - M1 motorway (Northern Ireland)
   - A194 road
   - 1p36 deletion syndrome
POV_check: EMPTY


In [12]:
import requests

HEADERS = {"User-Agent": "UniMannheim-SMDA-Project/1.0 (student research project)"}

def find_via_templates():
    # Instead of categories, search for pages that EMBED dispute templates
    # These are the actual template names Wikipedia uses
    templates = [
        "Template:POV",
        "Template:Disputed",
        "Template:Neutrality",
        "Template:Biased",
        "Template:Unbalanced",
    ]
    
    for template in templates:
        params = {
            "action": "query",
            "list": "embeddedin",   # pages that transclude this template
            "eititle": template,
            "eilimit": 5,
            "einamespace": 0,       # main articles only, no talk pages
            "format": "json"
        }
        r = requests.get("https://en.wikipedia.org/w/api.php", params=params, headers=HEADERS)
        pages = r.json()["query"]["embeddedin"]
        print(f"\n{template}: {len(pages)} pages found")
        for p in pages:
            print(f"   - {p['title']}")

find_via_templates()


Template:POV: 5 pages found
   - Apple Inc.
   - Anxiety
   - Alvin Toffler
   - Afterlife
   - The Buddha

Template:Disputed: 5 pages found
   - Amber Road
   - Baltic languages
   - Celtic music
   - History of Cuba
   - Conscription

Template:Neutrality: 5 pages found
   - Mandatory Swedish
   - Pennsylvania Railroad class T1
   - Baháʼí administration
   - Argentine Antarctica
   - Stan Lane

Template:Biased: 5 pages found
   - Pro-feminism
   - Remexido
   - Thomas Fersen
   - Nelson L. Goldberg
   - Sani Yakubu Rodi

Template:Unbalanced: 5 pages found
   - Anarcho-capitalism
   - Abdulaziz
   - FIDE
   - Hawaii
   - Ivanhoe


In [13]:
import requests

HEADERS = {"User-Agent": "UniMannheim-SMDA-Project/1.0 (student research project)"}

def fetch_contested_articles(limit_per_template=5):
    """Fetch contested articles using dispute templates"""
    templates = [
        "Template:POV",
        "Template:Disputed", 
        "Template:Neutrality",
        "Template:Biased",
        "Template:Unbalanced",
    ]
    
    contested = []
    seen = set()  # avoid duplicates
    
    for template in templates:
        params = {
            "action": "query",
            "list": "embeddedin",
            "eititle": template,
            "eilimit": limit_per_template,
            "einamespace": 0,
            "format": "json"
        }
        r = requests.get("https://en.wikipedia.org/w/api.php", params=params, headers=HEADERS)
        pages = r.json()["query"]["embeddedin"]
        for p in pages:
            if p["title"] not in seen:
                seen.add(p["title"])
                contested.append({"title": p["title"], "label": 0, "source_template": template})
    
    return contested


def fetch_stable_articles(limit=10):
    """Fetch stable (Featured) articles"""
    params = {
        "action": "query",
        "list": "categorymembers",
        "cmtitle": "Category:Featured_articles",
        "cmlimit": limit,
        "cmtype": "page",
        "format": "json"
    }
    r = requests.get("https://en.wikipedia.org/w/api.php", params=params, headers=HEADERS)
    pages = r.json()["query"]["categorymembers"]
    
    stable = []
    for p in pages:
        stable.append({"title": p["title"], "label": 1, "source_template": "Featured"})
    
    return stable


def fetch_article_text(title):
    """Fetch full text of one article"""
    params = {
        "action": "query",
        "titles": title,
        "prop": "revisions",
        "rvprop": "content",
        "rvslots": "main",
        "format": "json"
    }
    r = requests.get("https://en.wikipedia.org/w/api.php", params=params, headers=HEADERS)
    page = list(r.json()["query"]["pages"].values())[0]
    
    if "revisions" not in page:
        return None
    
    return page["revisions"][0]["slots"]["main"]["*"]


# ── Run everything ──────────────────────────────────────────

print("=" * 50)
print("CONTESTED ARTICLES")
print("=" * 50)
contested = fetch_contested_articles(limit_per_template=3)
for a in contested:
    print(f"  [{a['label']}] {a['title']}  (via {a['source_template']})")

print(f"\nTotal contested: {len(contested)}")

print("\n" + "=" * 50)
print("STABLE (FEATURED) ARTICLES")
print("=" * 50)
stable = fetch_stable_articles(limit=10)
for a in stable:
    print(f"  [{a['label']}] {a['title']}")

print(f"\nTotal stable: {len(stable)}")

print("\n" + "=" * 50)
print("SAMPLE ARTICLE TEXT FETCH")
print("=" * 50)
test_title = contested[0]["title"]
text = fetch_article_text(test_title)
if text:
    word_count = len(text.split())
    print(f"Article: '{test_title}'")
    print(f"Word count: {word_count}")
    print(f"First 200 chars: {text[:200]}")
else:
    print(f"Could not fetch '{test_title}'")

CONTESTED ARTICLES
  [0] Apple Inc.  (via Template:POV)
  [0] Anxiety  (via Template:POV)
  [0] Alvin Toffler  (via Template:POV)
  [0] Amber Road  (via Template:Disputed)
  [0] Baltic languages  (via Template:Disputed)
  [0] Celtic music  (via Template:Disputed)
  [0] Mandatory Swedish  (via Template:Neutrality)
  [0] Pennsylvania Railroad class T1  (via Template:Neutrality)
  [0] Baháʼí administration  (via Template:Neutrality)
  [0] Pro-feminism  (via Template:Biased)
  [0] Remexido  (via Template:Biased)
  [0] Thomas Fersen  (via Template:Biased)
  [0] Anarcho-capitalism  (via Template:Unbalanced)
  [0] Abdulaziz  (via Template:Unbalanced)
  [0] FIDE  (via Template:Unbalanced)

Total contested: 15

STABLE (FEATURED) ARTICLES
  [1] ? (2011 film)
  [1] 1 − 2 + 3 − 4 + ⋯
  [1] 1 Line (Sound Transit)
  [1] 1 Wall Street
  [1] 1st Cavalry Division (Kingdom of Yugoslavia)
  [1] 1st Missouri Field Battery
  [1] 1st Provisional Marine Brigade
  [1] 2nd Infantry Division (United Kingdom)
  

In [14]:
try:
    import spacy
    nlp = spacy.load("en_core_web_sm")
    doc = nlp("Some argue that climate change is allegedly disputed.")
    print("✅ spaCy working!")
    print(f"   Tokens: {[t.text for t in doc]}")
except Exception as e:
    print(f"❌ Error: {e}")

try:
    from lexicalrichness import LexicalRichness
    lex = LexicalRichness("The quick brown fox jumps over the lazy dog near the river bank")
    print(f"✅ lexicalrichness working! MTLD = {lex.mtld(threshold=0.72):.2f}")
except Exception as e:
    print(f"❌ Error: {e}")

✅ spaCy working!
   Tokens: ['Some', 'argue', 'that', 'climate', 'change', 'is', 'allegedly', 'disputed', '.']
✅ lexicalrichness working! MTLD = 23.66


In [15]:
import requests
import time

HEADERS = {"User-Agent": "UniMannheim-SMDA-Project/1.0 (student research project)"}

def count_template_articles(wiki, template):
    """Count how many articles embed a given template"""
    url = f"https://{wiki}.wikipedia.org/w/api.php"
    count = 0
    params = {
        "action": "query",
        "list": "embeddedin",
        "eititle": template,
        "eilimit": 500,
        "einamespace": 0,
        "format": "json"
    }
    while True:
        r = requests.get(url, params=params, headers=HEADERS)
        data = r.json()
        count += len(data["query"]["embeddedin"])
        if "continue" not in data:
            break
        params["eicontinue"] = data["continue"]["eicontinue"]
        time.sleep(0.5)
    return count

# English templates
en_templates = [
    "Template:POV",
    "Template:Disputed",
    "Template:Neutrality",
    "Template:Biased",
    "Template:Unbalanced",
    "Template:One-sided",
]

# German templates
de_templates = [
    "Vorlage:Neutralität",
    "Vorlage:Belege fehlen",
    "Vorlage:Einseitig",
    "Vorlage:Umstritten",
    "Vorlage:NPOV",
]

print("ENGLISH WIKIPEDIA — contested templates")
print("=" * 45)
en_total = 0
for t in en_templates:
    count = count_template_articles("en", t)
    en_total += count
    print(f"  {t.replace('Template:', '')}: {count} articles")
print(f"  TOTAL: {en_total}")

print("\nGERMAN WIKIPEDIA — contested templates")
print("=" * 45)
de_total = 0
for t in de_templates:
    count = count_template_articles("de", t)
    de_total += count
    print(f"  {t.replace('Vorlage:', '')}: {count} articles")
print(f"  TOTAL: {de_total}")

print("\nFEATURED ARTICLES (stable)")
print("=" * 45)
for wiki in ["en", "de"]:
    params = {
        "action": "query",
        "list": "categorymembers",
        "cmtitle": "Category:Featured_articles" if wiki == "en" else "Kategorie:Exzellenter_Artikel",
        "cmlimit": 500,
        "cmtype": "page",
        "format": "json"
    }
    r = requests.get(f"https://{wiki}.wikipedia.org/w/api.php", params=params, headers=HEADERS)
    count = len(r.json()["query"]["categorymembers"])
    print(f"  {wiki}.wikipedia Featured Articles (first page): {count}+")

ENGLISH WIKIPEDIA — contested templates
  POV: 2051 articles
  Disputed: 1704 articles
  Neutrality: 48 articles
  Biased: 13 articles
  Unbalanced: 521 articles
  One-sided: 0 articles
  TOTAL: 4337

GERMAN WIKIPEDIA — contested templates
  Neutralität: 1053 articles
  Belege fehlen: 51692 articles
  Einseitig: 0 articles
  Umstritten: 0 articles
  NPOV: 36 articles
  TOTAL: 52781

FEATURED ARTICLES (stable)
  en.wikipedia Featured Articles (first page): 500+
  de.wikipedia Featured Articles (first page): 0+


In [16]:
import requests
import time
import json
import os

HEADERS = {"User-Agent": "UniMannheim-SMDA-Project/1.0 (student research project)"}

def fetch_all_titles(wiki, source, source_type, limit=300):
    """
    Fetch article titles from a template (contested) or category (stable).
    source_type: 'template' or 'category'
    """
    url = f"https://{wiki}.wikipedia.org/w/api.php"
    titles = []
    
    if source_type == "template":
        params = {
            "action": "query",
            "list": "embeddedin",
            "eititle": source,
            "eilimit": 500,
            "einamespace": 0,
            "format": "json"
        }
        continue_key = "eicontinue"
        result_key = "embeddedin"
    else:  # category
        params = {
            "action": "query",
            "list": "categorymembers",
            "cmtitle": source,
            "cmlimit": 500,
            "cmtype": "page",
            "format": "json"
        }
        continue_key = "cmcontinue"
        result_key = "categorymembers"
    
    while len(titles) < limit:
        r = requests.get(url, params=params, headers=HEADERS)
        data = r.json()
        batch = data["query"][result_key]
        titles.extend([p["title"] for p in batch])
        
        if "continue" not in data or len(titles) >= limit:
            break
        params[continue_key] = data["continue"][continue_key]
        time.sleep(0.3)
    
    return titles[:limit]


def fetch_article_data(wiki, title):
    """
    Fetch full text + metadata for one article.
    Returns dict with all fields we need.
    """
    url = f"https://{wiki}.wikipedia.org/w/api.php"
    
    # Main article: text + edit stats
    params = {
        "action": "query",
        "titles": title,
        "prop": "revisions|info",
        "rvprop": "content|timestamp",
        "rvslots": "main",
        "rvlimit": 1,
        "inprop": "watched",
        "format": "json"
    }
    r = requests.get(url, params=params, headers=HEADERS)
    page = list(r.json()["query"]["pages"].values())[0]
    
    if "revisions" not in page:
        return None
    
    text = page["revisions"][0]["slots"]["main"]["*"]
    word_count = len(text.split())
    
    # Skip stubs
    if word_count < 500:
        return None
    
    # Edit history: total edits + unique editors
    params2 = {
        "action": "query",
        "titles": title,
        "prop": "revisions",
        "rvprop": "user|timestamp",
        "rvlimit": 500,
        "format": "json"
    }
    r2 = requests.get(url, params=params2, headers=HEADERS)
    page2 = list(r2.json()["query"]["pages"].values())[0]
    revisions = page2.get("revisions", [])
    unique_editors = len(set(rv.get("user", "") for rv in revisions))
    total_edits = len(revisions)
    first_edit = revisions[-1]["timestamp"] if revisions else None
    
    # Talk page: revert count + word count
    talk_title = f"Talk:{title}"
    params3 = {
        "action": "query",
        "titles": talk_title,
        "prop": "revisions",
        "rvprop": "comment|content",
        "rvslots": "main",
        "rvlimit": 500,
        "format": "json"
    }
    r3 = requests.get(url, params=params3, headers=HEADERS)
    talk_page = list(r3.json()["query"]["pages"].values())[0]
    talk_revisions = talk_page.get("revisions", [])
    
    revert_count = sum(
        1 for rv in talk_revisions
        if any(word in rv.get("comment", "").lower() 
               for word in ["revert", "reverted", "undid", "undo", "restored"])
    )
    
    talk_text = ""
    if talk_revisions and "slots" in talk_revisions[0]:
        talk_text = talk_revisions[0]["slots"]["main"].get("*", "")
    talk_word_count = len(talk_text.split())
    
    return {
        "title": title,
        "wiki": wiki,
        "text": text,
        "word_count": word_count,
        "total_edits": total_edits,
        "unique_editors": unique_editors,
        "first_edit": first_edit,
        "revert_count": revert_count,
        "talk_word_count": talk_word_count,
    }


def build_dataset(wiki, contested_sources, stable_source, target=300, out_file="dataset.json"):
    """
    Full pipeline: fetch titles → fetch data → save to JSON
    """
    dataset = []
    seen = set()

    # ── CONTESTED ──────────────────────────────
    print(f"\n[{wiki}] Fetching CONTESTED titles...")
    contested_titles = []
    per_source = target // len(contested_sources) + 10  # slight buffer
    
    for source in contested_sources:
        titles = fetch_all_titles(wiki, source, "template", limit=per_source)
        for t in titles:
            if t not in seen:
                seen.add(t)
                contested_titles.append(t)
        print(f"  {source}: {len(titles)} titles fetched (total so far: {len(contested_titles)})")
    
    contested_titles = contested_titles[:target]
    print(f"  → Using {len(contested_titles)} contested titles")

    print(f"[{wiki}] Fetching CONTESTED article data...")
    contested_done = 0
    for title in contested_titles:
        data = fetch_article_data(wiki, title)
        if data:
            data["label"] = 0
            data["label_name"] = "contested"
            dataset.append(data)
            contested_done += 1
            if contested_done % 10 == 0:
                print(f"  {contested_done}/{len(contested_titles)} contested done...")
        time.sleep(0.5)  # be polite to Wikipedia's servers

    # ── STABLE ─────────────────────────────────
    print(f"\n[{wiki}] Fetching STABLE titles...")
    stable_titles = []
    for t in fetch_all_titles(wiki, stable_source, "category", limit=target + 50):
        if t not in seen:
            seen.add(t)
            stable_titles.append(t)
    stable_titles = stable_titles[:target]
    print(f"  → Using {len(stable_titles)} stable titles")

    print(f"[{wiki}] Fetching STABLE article data...")
    stable_done = 0
    for title in stable_titles:
        data = fetch_article_data(wiki, title)
        if data:
            data["label"] = 1
            data["label_name"] = "stable"
            dataset.append(data)
            stable_done += 1
            if stable_done % 10 == 0:
                print(f"  {stable_done}/{len(stable_titles)} stable done...")
        time.sleep(0.5)

    # ── SAVE ───────────────────────────────────
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(dataset, f, ensure_ascii=False, indent=2)
    
    actual_contested = sum(1 for d in dataset if d["label"] == 0)
    actual_stable = sum(1 for d in dataset if d["label"] == 1)
    print(f"\n✅ Saved {len(dataset)} articles to {out_file}")
    print(f"   Contested: {actual_contested} | Stable: {actual_stable}")
    
    return dataset


# ── RUN ────────────────────────────────────────────────────

# First fix German featured articles category name
print("Checking German Featured Articles category...")
for cat_name in ["Kategorie:Exzellenter Artikel", "Kategorie:Wikipedia:Exzellent"]:
    params = {
        "action": "query",
        "list": "categorymembers",
        "cmtitle": cat_name,
        "cmlimit": 5,
        "cmtype": "page",
        "format": "json"
    }
    r = requests.get("https://de.wikipedia.org/w/api.php", params=params, headers=HEADERS)
    results = r.json()["query"]["categorymembers"]
    print(f"  '{cat_name}': {len(results)} results")
    if results:
        for a in results:
            print(f"    - {a['title']}")

Checking German Featured Articles category...
  'Kategorie:Exzellenter Artikel': 0 results
  'Kategorie:Wikipedia:Exzellent': 5 results
    - Wikipedia:Exzellente Artikel
    - 4th Infantry Division (Vereinigte Staaten)
    - 4 Monate, 3 Wochen und 2 Tage
    - 7. Sinfonie (Bruckner)
    - 24 (Fernsehserie)


In [18]:
import requests
import time
import json

HEADERS = {"User-Agent": "UniMannheim-SMDA-Project/1.0 (student research project)"}

def safe_get(url, params, retries=3, wait=2):
    """Wrapper around requests.get with retries on failure"""
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=15)
            if r.status_code == 200 and r.text.strip():
                return r.json()
            else:
                print(f"  ⚠️ Empty response (attempt {attempt+1}), retrying...")
        except Exception as e:
            print(f"  ⚠️ Error: {e} (attempt {attempt+1}), retrying...")
        time.sleep(wait * (attempt + 1))  # wait longer each retry
    return None  # give up after retries


def fetch_all_titles(wiki, source, source_type, limit=300):
    url = f"https://{wiki}.wikipedia.org/w/api.php"
    titles = []

    if source_type == "template":
        params = {
            "action": "query",
            "list": "embeddedin",
            "eititle": source,
            "eilimit": 500,
            "einamespace": 0,
            "format": "json"
        }
        continue_key = "eicontinue"
        result_key = "embeddedin"
    else:
        params = {
            "action": "query",
            "list": "categorymembers",
            "cmtitle": source,
            "cmlimit": 500,
            "cmtype": "page",
            "format": "json"
        }
        continue_key = "cmcontinue"
        result_key = "categorymembers"

    while len(titles) < limit:
        data = safe_get(url, params)
        if not data:
            print(f"  ❌ Giving up on {source} after failed retries")
            break
        batch = data["query"][result_key]
        titles.extend([p["title"] for p in batch])
        if "continue" not in data or len(titles) >= limit:
            break
        params[continue_key] = data["continue"][continue_key]
        time.sleep(0.5)

    return titles[:limit]


def fetch_article_data(wiki, title):
    url = f"https://{wiki}.wikipedia.org/w/api.php"

    # Main article text
    data = safe_get(url, {
        "action": "query",
        "titles": title,
        "prop": "revisions|info",
        "rvprop": "content|timestamp",
        "rvslots": "main",
        "rvlimit": 1,
        "format": "json"
    })
    if not data:
        return None

    page = list(data["query"]["pages"].values())[0]
    if "revisions" not in page:
        return None

    text = page["revisions"][0]["slots"]["main"]["*"]
    word_count = len(text.split())
    if word_count < 500:
        return None

    # Edit history
    data2 = safe_get(url, {
        "action": "query",
        "titles": title,
        "prop": "revisions",
        "rvprop": "user|timestamp",
        "rvlimit": 500,
        "format": "json"
    })
    revisions = []
    if data2:
        page2 = list(data2["query"]["pages"].values())[0]
        revisions = page2.get("revisions", [])

    unique_editors = len(set(rv.get("user", "") for rv in revisions))
    total_edits = len(revisions)
    first_edit = revisions[-1]["timestamp"] if revisions else None

    # Talk page
    data3 = safe_get(url, {
        "action": "query",
        "titles": f"Talk:{title}",
        "prop": "revisions",
        "rvprop": "comment|content",
        "rvslots": "main",
        "rvlimit": 500,
        "format": "json"
    })
    revert_count = 0
    talk_word_count = 0
    if data3:
        talk_page = list(data3["query"]["pages"].values())[0]
        talk_revisions = talk_page.get("revisions", [])
        revert_count = sum(
            1 for rv in talk_revisions
            if any(w in rv.get("comment", "").lower()
                   for w in ["revert", "reverted", "undid", "undo", "restored"])
        )
        if talk_revisions and "slots" in talk_revisions[0]:
            talk_text = talk_revisions[0]["slots"]["main"].get("*", "")
            talk_word_count = len(talk_text.split())

    return {
        "title": title,
        "wiki": wiki,
        "text": text,
        "word_count": word_count,
        "total_edits": total_edits,
        "unique_editors": unique_editors,
        "first_edit": first_edit,
        "revert_count": revert_count,
        "talk_word_count": talk_word_count,
    }


def build_dataset(wiki, contested_sources, stable_source, target=300, out_file="dataset.json"):
    dataset = []
    seen = set()

    # CONTESTED
    print(f"\n[{wiki}] Fetching CONTESTED titles...")
    contested_titles = []
    per_source = target // len(contested_sources) + 10

    for source in contested_sources:
        titles = fetch_all_titles(wiki, source, "template", limit=per_source)
        for t in titles:
            if t not in seen:
                seen.add(t)
                contested_titles.append(t)
        print(f"  {source}: {len(titles)} fetched (total: {len(contested_titles)})")

    contested_titles = contested_titles[:target]
    print(f"  → Using {len(contested_titles)} contested titles")

    print(f"[{wiki}] Fetching CONTESTED article data (this takes a while)...")
    contested_done = 0
    for title in contested_titles:
        data = fetch_article_data(wiki, title)
        if data:
            data["label"] = 0
            data["label_name"] = "contested"
            dataset.append(data)
            contested_done += 1
            if contested_done % 10 == 0:
                print(f"  {contested_done}/{len(contested_titles)} contested done...")
        time.sleep(0.5)

    # STABLE
    print(f"\n[{wiki}] Fetching STABLE titles...")
    stable_titles = []
    for t in fetch_all_titles(wiki, stable_source, "category", limit=target + 50):
        if t not in seen:
            seen.add(t)
            stable_titles.append(t)
    stable_titles = stable_titles[:target]
    print(f"  → Using {len(stable_titles)} stable titles")

    print(f"[{wiki}] Fetching STABLE article data...")
    stable_done = 0
    for title in stable_titles:
        data = fetch_article_data(wiki, title)
        if data:
            data["label"] = 1
            data["label_name"] = "stable"
            dataset.append(data)
            stable_done += 1
            if stable_done % 10 == 0:
                print(f"  {stable_done}/{len(stable_titles)} stable done...")
        time.sleep(0.5)

    # SAVE — save after each language so nothing is lost
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(dataset, f, ensure_ascii=False, indent=2)

    actual_contested = sum(1 for d in dataset if d["label"] == 0)
    actual_stable    = sum(1 for d in dataset if d["label"] == 1)
    print(f"\n✅ Saved {len(dataset)} articles → {out_file}")
    print(f"   Contested: {actual_contested} | Stable: {actual_stable}")
    return dataset


# ── RUN ──────────────────────────────────────────────────

build_dataset(
    wiki="en",
    contested_sources=["Template:POV","Template:Disputed","Template:Neutrality","Template:Biased","Template:Unbalanced"],
    stable_source="Category:Featured_articles",
    target=300,
    out_file="dataset_en.json"
)

build_dataset(
    wiki="de",
    contested_sources=["Vorlage:Neutralität","Vorlage:NPOV"],
    stable_source="Kategorie:Wikipedia:Exzellent",
    target=300,
    out_file="dataset_de.json"
)

print("\n🎉 Both datasets complete!")


[en] Fetching CONTESTED titles...
  Template:POV: 70 fetched (total: 70)
  Template:Disputed: 70 fetched (total: 140)
  Template:Neutrality: 48 fetched (total: 188)
  Template:Biased: 13 fetched (total: 201)
  Template:Unbalanced: 70 fetched (total: 271)
  → Using 271 contested titles
[en] Fetching CONTESTED article data (this takes a while)...
  10/271 contested done...
  20/271 contested done...
  30/271 contested done...
  40/271 contested done...
  50/271 contested done...
  60/271 contested done...
  70/271 contested done...
  80/271 contested done...
  90/271 contested done...
  100/271 contested done...
  110/271 contested done...
  120/271 contested done...
  130/271 contested done...
  140/271 contested done...
  150/271 contested done...
  160/271 contested done...
  ⚠️ Empty response (attempt 1), retrying...
  ⚠️ Empty response (attempt 2), retrying...
  ⚠️ Empty response (attempt 3), retrying...
  ⚠️ Empty response (attempt 1), retrying...
  ⚠️ Empty response (attempt 2), 

In [ ]:
# Quick sanity check after it finishes
for fname in ["dataset_en.json", "dataset_de.json"]:
    with open(fname, "r") as f:
        data = json.load(f)
    contested = [d for d in data if d["label"] == 0]
    stable    = [d for d in data if d["label"] == 1]
    avg_words = sum(d["word_count"] for d in data) / len(data)
    print(f"\n{fname}")
    print(f"  Contested : {len(contested)}")
    print(f"  Stable    : {len(stable)}")
    print(f"  Avg words : {avg_words:.0f}")
    print(f"  Sample contested: {contested[0]['title']}")
    print(f"  Sample stable   : {stable[0]['title']}")

In [20]:
import requests
import time
import json
import random

HEADERS = {"User-Agent": "UniMannheim-SMDA-Project/1.0 (student research project)"}

def safe_get(url, params, retries=5, base_wait=3):
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=20)
            if r.status_code == 429:
                wait = 60  # rate limited — wait a full minute
                print(f"  🚦 Rate limited! Waiting {wait}s...")
                time.sleep(wait)
                continue
            if r.status_code == 200 and r.text.strip():
                return r.json()
            else:
                wait = base_wait * (attempt + 1)
                print(f"  ⚠️ Empty response (attempt {attempt+1}), waiting {wait}s...")
                time.sleep(wait)
        except Exception as e:
            wait = base_wait * (attempt + 1)
            print(f"  ⚠️ Error: {e} (attempt {attempt+1}), waiting {wait}s...")
            time.sleep(wait)
    return None


def fetch_all_titles(wiki, source, source_type, limit=300):
    url = f"https://{wiki}.wikipedia.org/w/api.php"
    titles = []

    if source_type == "template":
        params = {
            "action": "query",
            "list": "embeddedin",
            "eititle": source,
            "eilimit": 500,
            "einamespace": 0,
            "format": "json"
        }
        continue_key = "eicontinue"
        result_key = "embeddedin"
    else:
        params = {
            "action": "query",
            "list": "categorymembers",
            "cmtitle": source,
            "cmlimit": 500,
            "cmtype": "page",
            "format": "json"
        }
        continue_key = "cmcontinue"
        result_key = "categorymembers"

    while len(titles) < limit:
        data = safe_get(url, params)
        if not data:
            print(f"  ❌ Giving up on {source}")
            break
        batch = data["query"][result_key]
        titles.extend([p["title"] for p in batch])
        if "continue" not in data or len(titles) >= limit:
            break
        params[continue_key] = data["continue"][continue_key]
        time.sleep(1)  # pause between pagination calls

    return titles[:limit]


def fetch_article_data(wiki, title):
    url = f"https://{wiki}.wikipedia.org/w/api.php"

    # Main article text
    data = safe_get(url, {
        "action": "query",
        "titles": title,
        "prop": "revisions|info",
        "rvprop": "content|timestamp",
        "rvslots": "main",
        "rvlimit": 1,
        "format": "json"
    })
    if not data:
        return None

    page = list(data["query"]["pages"].values())[0]
    if "revisions" not in page:
        return None

    text = page["revisions"][0]["slots"]["main"]["*"]
    word_count = len(text.split())
    if word_count < 500:
        return None

    time.sleep(1)  # pause between calls for same article

    # Edit history
    data2 = safe_get(url, {
        "action": "query",
        "titles": title,
        "prop": "revisions",
        "rvprop": "user|timestamp",
        "rvlimit": 500,
        "format": "json"
    })
    revisions = []
    if data2:
        page2 = list(data2["query"]["pages"].values())[0]
        revisions = page2.get("revisions", [])

    unique_editors = len(set(rv.get("user", "") for rv in revisions))
    total_edits = len(revisions)
    first_edit = revisions[-1]["timestamp"] if revisions else None

    time.sleep(1)

    # Talk page
    data3 = safe_get(url, {
        "action": "query",
        "titles": f"Talk:{title}",
        "prop": "revisions",
        "rvprop": "comment|content",
        "rvslots": "main",
        "rvlimit": 500,
        "format": "json"
    })
    revert_count = 0
    talk_word_count = 0
    if data3:
        talk_page = list(data3["query"]["pages"].values())[0]
        talk_revisions = talk_page.get("revisions", [])
        revert_count = sum(
            1 for rv in talk_revisions
            if any(w in rv.get("comment", "").lower()
                   for w in ["revert", "reverted", "undid", "undo", "restored"])
        )
        if talk_revisions and "slots" in talk_revisions[0]:
            talk_text = talk_revisions[0]["slots"]["main"].get("*", "")
            talk_word_count = len(talk_text.split())

    return {
        "title": title,
        "wiki": wiki,
        "text": text,
        "word_count": word_count,
        "total_edits": total_edits,
        "unique_editors": unique_editors,
        "first_edit": first_edit,
        "revert_count": revert_count,
        "talk_word_count": talk_word_count,
    }


def build_dataset(wiki, contested_sources, stable_source, target=300,
                  out_file="dataset.json", existing_file=None):
    """
    Build dataset. If existing_file is provided, load it and skip
    already-fetched articles (allows resuming after rate limit).
    """
    dataset = []
    seen = set()

    # Load existing data if resuming
    if existing_file:
        try:
            with open(existing_file, "r", encoding="utf-8") as f:
                dataset = json.load(f)
            seen = set(d["title"] for d in dataset)
            already_contested = sum(1 for d in dataset if d["label"] == 0)
            already_stable = sum(1 for d in dataset if d["label"] == 1)
            print(f"  📂 Resuming from {existing_file}")
            print(f"     Already have: {already_contested} contested, {already_stable} stable")
        except FileNotFoundError:
            print(f"  No existing file found, starting fresh")

    already_contested = sum(1 for d in dataset if d["label"] == 0)
    already_stable = sum(1 for d in dataset if d["label"] == 1)
    need_contested = target - already_contested
    need_stable = target - already_stable

    # CONTESTED
    if need_contested > 0:
        print(f"\n[{wiki}] Fetching CONTESTED titles (need {need_contested} more)...")
        contested_titles = []
        per_source = (need_contested // len(contested_sources)) + 20

        for source in contested_sources:
            titles = fetch_all_titles(wiki, source, "template", limit=per_source + 50)
            for t in titles:
                if t not in seen:
                    contested_titles.append(t)
            print(f"  {source}: {len(titles)} fetched")

        contested_titles = contested_titles[:need_contested + 20]
        print(f"  → Fetching data for {len(contested_titles)} articles...")

        done = 0
        for title in contested_titles:
            if already_contested + done >= target:
                break
            if title in seen:
                continue
            # Random pause between 1.5–3s to avoid rate limiting
            time.sleep(random.uniform(1.5, 3.0))
            data = fetch_article_data(wiki, title)
            if data:
                data["label"] = 0
                data["label_name"] = "contested"
                dataset.append(data)
                seen.add(title)
                done += 1
                if done % 10 == 0:
                    # Save progress every 10 articles
                    with open(out_file, "w", encoding="utf-8") as f:
                        json.dump(dataset, f, ensure_ascii=False, indent=2)
                    print(f"  {done} new contested done (saved)...")

    # STABLE
    if need_stable > 0:
        print(f"\n[{wiki}] Fetching STABLE titles (need {need_stable} more)...")
        stable_titles = []
        for t in fetch_all_titles(wiki, stable_source, "category", limit=target + 50):
            if t not in seen:
                stable_titles.append(t)
        stable_titles = stable_titles[:need_stable + 20]
        print(f"  → Fetching data for {len(stable_titles)} articles...")

        done = 0
        for title in stable_titles:
            if already_stable + done >= target:
                break
            time.sleep(random.uniform(1.5, 3.0))
            data = fetch_article_data(wiki, title)
            if data:
                data["label"] = 1
                data["label_name"] = "stable"
                dataset.append(data)
                seen.add(title)
                done += 1
                if done % 10 == 0:
                    with open(out_file, "w", encoding="utf-8") as f:
                        json.dump(dataset, f, ensure_ascii=False, indent=2)
                    print(f"  {done} new stable done (saved)...")

    # Final save
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(dataset, f, ensure_ascii=False, indent=2)

    final_contested = sum(1 for d in dataset if d["label"] == 0)
    final_stable = sum(1 for d in dataset if d["label"] == 1)
    print(f"\n✅ Saved {len(dataset)} articles → {out_file}")
    print(f"   Contested: {final_contested} | Stable: {final_stable}")
    return dataset


# ── RUN — resume English from where we left off ──────────────

print("Starting English dataset (resuming)...")
build_dataset(
    wiki="en",
    contested_sources=["Template:POV","Template:Disputed","Template:Neutrality",
                       "Template:Biased","Template:Unbalanced"],
    stable_source="Category:Featured_articles",
    target=300,
    out_file="dataset_en.json",
    existing_file="dataset_en.json"  # resume from saved progress
)

print("\nStarting German dataset...")
build_dataset(
    wiki="de",
    contested_sources=["Vorlage:Neutralität","Vorlage:NPOV"],
    stable_source="Kategorie:Wikipedia:Exzellent",
    target=300,
    out_file="dataset_de.json"
)

print("\n🎉 Both datasets complete!")

Starting English dataset (resuming)...
  📂 Resuming from dataset_en.json
     Already have: 163 contested, 0 stable

[en] Fetching CONTESTED titles (need 137 more)...
  🚦 Rate limited! Waiting 60s...
  🚦 Rate limited! Waiting 60s...
  🚦 Rate limited! Waiting 60s...
  🚦 Rate limited! Waiting 60s...
  🚦 Rate limited! Waiting 60s...


KeyboardInterrupt: 

In [52]:
import requests

HEADERS = {"User-Agent": "UniMannheim-SMDA-Project/1.0 (student research project)"}

r = requests.get("https://en.wikipedia.org/w/api.php", 
                 params={"action": "query", "list": "embeddedin",
                         "eititle": "Template:POV", "eilimit": 3,
                         "einamespace": 0, "format": "json"},
                 headers=HEADERS)

print(f"Status: {r.status_code}")
print(f"Response: {r.text[:200]}")

Status: 200
Response: {"batchcomplete":"","continue":{"eicontinue":"0|1178","continue":"-||"},"query":{"embeddedin":[{"pageid":856,"ns":0,"title":"Apple Inc."},{"pageid":922,"ns":0,"title":"Anxiety"},{"pageid":930,"ns":0,"


In [58]:
import requests
import time
import json
import random

HEADERS = {"User-Agent": "UniMannheim-SMDA-Project/1.0 (student research project)"}

def safe_get(url, params, retries=5, base_wait=3):
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=20)
            if r.status_code == 429:
                wait = 60  # rate limited — wait a full minute
                print(f"  🚦 Rate limited! Waiting {wait}s...")
                time.sleep(wait)
                continue
            if r.status_code == 200 and r.text.strip():
                return r.json()
            else:
                wait = base_wait * (attempt + 1)
                print(f"  ⚠️ Empty response (attempt {attempt+1}), waiting {wait}s...")
                time.sleep(wait)
        except Exception as e:
            wait = base_wait * (attempt + 1)
            print(f"  ⚠️ Error: {e} (attempt {attempt+1}), waiting {wait}s...")
            time.sleep(wait)
    return None


def fetch_all_titles(wiki, source, source_type, limit=300):
    url = f"https://{wiki}.wikipedia.org/w/api.php"
    titles = []

    if source_type == "template":
        params = {
            "action": "query",
            "list": "embeddedin",
            "eititle": source,
            "eilimit": 500,
            "einamespace": 0,
            "format": "json"
        }
        continue_key = "eicontinue"
        result_key = "embeddedin"
    else:
        params = {
            "action": "query",
            "list": "categorymembers",
            "cmtitle": source,
            "cmlimit": 500,
            "cmtype": "page",
            "format": "json"
        }
        continue_key = "cmcontinue"
        result_key = "categorymembers"

    while len(titles) < limit:
        data = safe_get(url, params)
        if not data:
            print(f"  ❌ Giving up on {source}")
            break
        batch = data["query"][result_key]
        titles.extend([p["title"] for p in batch])
        if "continue" not in data or len(titles) >= limit:
            break
        params[continue_key] = data["continue"][continue_key]
        time.sleep(1)  # pause between pagination calls

    return titles[:limit]


def fetch_article_data(wiki, title):
    url = f"https://{wiki}.wikipedia.org/w/api.php"

    # Main article text
    data = safe_get(url, {
        "action": "query",
        "titles": title,
        "prop": "revisions|info",
        "rvprop": "content|timestamp",
        "rvslots": "main",
        "rvlimit": 1,
        "format": "json"
    })
    if not data:
        return None

    page = list(data["query"]["pages"].values())[0]
    if "revisions" not in page:
        return None

    text = page["revisions"][0]["slots"]["main"]["*"]
    word_count = len(text.split())
    if word_count < 500:
        return None

    time.sleep(1)  # pause between calls for same article

    # Edit history
    data2 = safe_get(url, {
        "action": "query",
        "titles": title,
        "prop": "revisions",
        "rvprop": "user|timestamp",
        "rvlimit": 500,
        "format": "json"
    })
    revisions = []
    if data2:
        page2 = list(data2["query"]["pages"].values())[0]
        revisions = page2.get("revisions", [])

    unique_editors = len(set(rv.get("user", "") for rv in revisions))
    total_edits = len(revisions)
    first_edit = revisions[-1]["timestamp"] if revisions else None

    time.sleep(1)

    # Talk page
    data3 = safe_get(url, {
        "action": "query",
        "titles": f"Talk:{title}",
        "prop": "revisions",
        "rvprop": "comment|content",
        "rvslots": "main",
        "rvlimit": 500,
        "format": "json"
    })
    revert_count = 0
    talk_word_count = 0
    if data3:
        talk_page = list(data3["query"]["pages"].values())[0]
        talk_revisions = talk_page.get("revisions", [])
        revert_count = sum(
            1 for rv in talk_revisions
            if any(w in rv.get("comment", "").lower()
                   for w in ["revert", "reverted", "undid", "undo", "restored"])
        )
        if talk_revisions and "slots" in talk_revisions[0]:
            talk_text = talk_revisions[0]["slots"]["main"].get("*", "")
            talk_word_count = len(talk_text.split())

    return {
        "title": title,
        "wiki": wiki,
        "text": text,
        "word_count": word_count,
        "total_edits": total_edits,
        "unique_editors": unique_editors,
        "first_edit": first_edit,
        "revert_count": revert_count,
        "talk_word_count": talk_word_count,
    }


def build_dataset(wiki, contested_sources, stable_source, target=300,
                  out_file="dataset.json", existing_file=None):
    """
    Build dataset. If existing_file is provided, load it and skip
    already-fetched articles (allows resuming after rate limit).
    """
    dataset = []
    seen = set()

    # Load existing data if resuming
    if existing_file:
        try:
            with open(existing_file, "r", encoding="utf-8") as f:
                dataset = json.load(f)
            seen = set(d["title"] for d in dataset)
            already_contested = sum(1 for d in dataset if d["label"] == 0)
            already_stable = sum(1 for d in dataset if d["label"] == 1)
            print(f"  📂 Resuming from {existing_file}")
            print(f"     Already have: {already_contested} contested, {already_stable} stable")
        except FileNotFoundError:
            print(f"  No existing file found, starting fresh")

    already_contested = sum(1 for d in dataset if d["label"] == 0)
    already_stable = sum(1 for d in dataset if d["label"] == 1)
    need_contested = target - already_contested
    need_stable = target - already_stable

    # CONTESTED
    if need_contested > 0:
        print(f"\n[{wiki}] Fetching CONTESTED titles (need {need_contested} more)...")
        contested_titles = []
        per_source = (need_contested // len(contested_sources)) + 20

        for source in contested_sources:
            titles = fetch_all_titles(wiki, source, "template", limit=per_source + 50)
            for t in titles:
                if t not in seen:
                    contested_titles.append(t)
            print(f"  {source}: {len(titles)} fetched")

        contested_titles = contested_titles[:need_contested + 20]
        print(f"  → Fetching data for {len(contested_titles)} articles...")

        done = 0
        for title in contested_titles:
            if already_contested + done >= target:
                break
            if title in seen:
                continue
            # Random pause between 1.5–3s to avoid rate limiting
            time.sleep(random.uniform(1.5, 3.0))
            data = fetch_article_data(wiki, title)
            if data:
                data["label"] = 0
                data["label_name"] = "contested"
                dataset.append(data)
                seen.add(title)
                done += 1
                if done % 10 == 0:
                    # Save progress every 10 articles
                    with open(out_file, "w", encoding="utf-8") as f:
                        json.dump(dataset, f, ensure_ascii=False, indent=2)
                    print(f"  {done} new contested done (saved)...")

    # STABLE
    if need_stable > 0:
        print(f"\n[{wiki}] Fetching STABLE titles (need {need_stable} more)...")
        stable_titles = []
        for t in fetch_all_titles(wiki, stable_source, "category", limit=target + 50):
            if t not in seen:
                stable_titles.append(t)
        stable_titles = stable_titles[:need_stable + 20]
        print(f"  → Fetching data for {len(stable_titles)} articles...")

        done = 0
        for title in stable_titles:
            if already_stable + done >= target:
                break
            time.sleep(random.uniform(1.5, 3.0))
            data = fetch_article_data(wiki, title)
            if data:
                data["label"] = 1
                data["label_name"] = "stable"
                dataset.append(data)
                seen.add(title)
                done += 1
                if done % 10 == 0:
                    with open(out_file, "w", encoding="utf-8") as f:
                        json.dump(dataset, f, ensure_ascii=False, indent=2)
                    print(f"  {done} new stable done (saved)...")

    # Final save
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(dataset, f, ensure_ascii=False, indent=2)

    final_contested = sum(1 for d in dataset if d["label"] == 0)
    final_stable = sum(1 for d in dataset if d["label"] == 1)
    print(f"\n✅ Saved {len(dataset)} articles → {out_file}")
    print(f"   Contested: {final_contested} | Stable: {final_stable}")
    return dataset


# ── RUN — resume English from where we left off ──────────────

print("Starting English dataset (resuming)...")
build_dataset(
    wiki="en",
    contested_sources=["Template:POV","Template:Disputed","Template:Neutrality",
                       "Template:Biased","Template:Unbalanced"],
    stable_source="Category:Featured_articles",
    target=300,
    out_file="dataset_en.json",
    existing_file="dataset_en.json"  # resume from saved progress
)

print("\nStarting German dataset...")
build_dataset(
    wiki="de",
    contested_sources=["Vorlage:Neutralität","Vorlage:NPOV"],
    stable_source="Kategorie:Wikipedia:Exzellent",
    target=300,
    out_file="dataset_de.json",
    existing_file="dataset_de.json"
)

print("\n🎉 Both datasets complete!")

Starting English dataset (resuming)...
  📂 Resuming from dataset_en.json
     Already have: 300 contested, 300 stable

✅ Saved 600 articles → dataset_en.json
   Contested: 300 | Stable: 300

Starting German dataset...
  📂 Resuming from dataset_de.json
     Already have: 178 contested, 30 stable

[de] Fetching CONTESTED titles (need 122 more)...
  Vorlage:Neutralität: 131 fetched
  Vorlage:NPOV: 36 fetched
  → Fetching data for 4 articles...

[de] Fetching STABLE titles (need 270 more)...
  → Fetching data for 290 articles...
  10 new stable done (saved)...
  20 new stable done (saved)...
  30 new stable done (saved)...
  40 new stable done (saved)...
  50 new stable done (saved)...
  60 new stable done (saved)...
  70 new stable done (saved)...
  80 new stable done (saved)...
  90 new stable done (saved)...
  100 new stable done (saved)...
  110 new stable done (saved)...
  120 new stable done (saved)...
  130 new stable done (saved)...
  140 new stable done (saved)...
  150 new stable

In [ ]:
# Step 2 — test fetching trends for a few of your actual article titles
from pytrends.request import TrendReq
import time

pytrends = TrendReq(hl='en-US', tz=360)

# Use 3 articles from your dataset — one contested, one stable, one in between
test_titles = ["Apple Inc.", "Anxiety", "1 Wall Street"]

for title in test_titles:
    try:
        pytrends.build_payload([title], timeframe='today 12-m', geo='')
        data = pytrends.interest_over_time()
        
        if data.empty:
            print(f"⚠️  '{title}': no data returned")
        else:
            values = data[title].tolist()
            avg = sum(values) / len(values)
            peak = max(values)
            print(f"✅ '{title}':")
            print(f"   Avg interest (12 months): {avg:.1f}")
            print(f"   Peak interest: {peak}")
            print(f"   Data points: {len(values)}")
    except Exception as e:
        print(f"❌ '{title}': {e}")
    
    time.sleep(3) 

/opt/anaconda3/envs/thesis/lib/python3.10/site-packages/pytrends/request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)


✅ 'Apple Inc.':
   Avg interest (12 months): 29.4
   Peak interest: 100
   Data points: 54


/opt/anaconda3/envs/thesis/lib/python3.10/site-packages/pytrends/request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)


✅ 'Anxiety':
   Avg interest (12 months): 58.3
   Peak interest: 100
   Data points: 54


/opt/anaconda3/envs/thesis/lib/python3.10/site-packages/pytrends/request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)


✅ '1 Wall Street':
   Avg interest (12 months): 30.5
   Peak interest: 100
   Data points: 54


In [5]:
from pytrends.request import TrendReq
import json
import time
import random

pytrends = TrendReq(hl='en-US', tz=360)

def get_trends(title, geo=''):
    """
    Fetch Google Trends data for a title.
    Returns avg_interest, peak_interest, volatility (std dev)
    geo='' means worldwide, geo='DE' means Germany only
    """
    try:
        pytrends.build_payload([title], timeframe='today 12-m', geo=geo)
        data = pytrends.interest_over_time()
        
        if data.empty or title not in data.columns:
            return None
        
        values = data[title].tolist()
        avg   = round(sum(values) / len(values), 2)
        peak  = max(values)
        std   = round((sum((v - avg)**2 for v in values) / len(values))**0.5, 2)
        
        return {
            "trends_avg": avg,       # stable long-term interest
            "trends_peak": peak,     # highest single spike
            "trends_std": std,       # volatility — high = trending topic, low = evergreen
        }
    except Exception as e:
        print(f"  ⚠️ Trends error for '{title}': {e}")
        return None


def enrich_dataset_with_trends(input_file, output_file, geo=''):
    """Add Google Trends data to every article in the dataset"""
    
    with open(input_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    print(f"Enriching {len(data)} articles with Google Trends...")
    print(f"Geo: '{geo}' (empty = worldwide)\n")
    
    enriched = 0
    skipped  = 0
    
    for i, article in enumerate(data):
        # Skip if already enriched
        if "trends_avg" in article:
            enriched += 1
            continue
        
        title = article["title"]
        trends = get_trends(title, geo=geo)
        
        if trends:
            article["trends_avg"]  = trends["trends_avg"]
            article["trends_peak"] = trends["trends_peak"]
            article["trends_std"]  = trends["trends_std"]
            enriched += 1
            print(f"  [{i+1}/{len(data)}] ✅ '{title}' — avg: {trends['trends_avg']}, std: {trends['trends_std']}")
        else:
            article["trends_avg"]  = None
            article["trends_peak"] = None
            article["trends_std"]  = None
            skipped += 1
            print(f"  [{i+1}/{len(data)}] ⚠️  '{title}' — no data")
        
        # Save every 20 articles so nothing is lost
        if (i + 1) % 20 == 0:
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False, indent=2)
            print(f"  💾 Progress saved ({i+1}/{len(data)})\n")
        
        # Random delay to avoid Google rate limiting
        time.sleep(random.uniform(3.0, 6.0))
    
    # Final save
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    
    print(f"\n✅ Done! Saved to {output_file}")
    print(f"   Enriched: {enriched} | No data: {skipped}")
    return data


# ── RUN ──────────────────────────────────────────────────────

# First test on just 5 articles before running the full thing
print("=== TEST RUN (5 articles) ===")

with open("dataset_en.json", "r") as f:
    sample = json.load(f)[:5]

for article in sample:
    title = article["title"]
    trends = get_trends(title)
    label = "contested" if article["label"] == 0 else "stable"
    if trends:
        print(f"[{label}] '{title}'")
        print(f"   avg={trends['trends_avg']}, peak={trends['trends_peak']}, std={trends['trends_std']}")
    else:
        print(f"[{label}] '{title}' — no data")
    time.sleep(4)

print("\n=== If test looks good, run the full enrichment: ===")
print("enrich_dataset_with_trends('dataset_en.json', 'dataset_en_trends.json')")

=== TEST RUN (5 articles) ===


/opt/anaconda3/envs/thesis/lib/python3.10/site-packages/pytrends/request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)


[contested] 'Apple Inc.'
   avg=29.44, peak=100, std=22.49


/opt/anaconda3/envs/thesis/lib/python3.10/site-packages/pytrends/request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)


[contested] 'Anxiety'
   avg=58.26, peak=100, std=11.95


...

[contested] 'Alvin Toffler'
   avg=51.85, peak=100, std=15.23


/opt/anaconda3/envs/thesis/lib/python3.10/site-packages/pytrends/request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)


[contested] 'Afterlife'
   avg=52.76, peak=100, std=10.38


/opt/anaconda3/envs/thesis/lib/python3.10/site-packages/pytrends/request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)


[contested] 'The Buddha'
   avg=56.15, peak=100, std=12.32

=== If test looks good, run the full enrichment: ===
enrich_dataset_with_trends('dataset_en.json', 'dataset_en_trends.json')
